<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 00 — Environment Check

Run this first, top to bottom; there is nothing to write. It checks that the
tools and `Ex_6_core.py` work on your machine, and shows the one calling
convention you have not met yet, L-BFGS's closure. If a cell fails, fix it
before going on.

---

## 0 · Versions

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import sys
import numpy as np
import matplotlib
import scipy
import torch
import torch.nn as nn

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("scipy      ", scipy.__version__, " (notebook 04's linear program)")
print("torch      ", torch.__version__)
print("cuda        available:", torch.cuda.is_available(), " (not needed)")

**What you should see.** Five version numbers and `cuda available: False` on
most machines. Any Python from 3.9 and any PyTorch from 2.0 will do. **No GPU is
required anywhere in Ex_06.**

---

## 1 · The shared module and its three datasets

In [ ]:
import Ex_6_core as core

print("module file:", core.__file__)
print("output dir :", core.OUTPUT_DIR)
print()

x_cal, y_cal = core.calibration_dataset()
X_vib, y_vib = core.vibration_dataset()
x_res, y_res = core.response_dataset()

print("calibration line   :", x_cal.shape, "  true slope", core.TRUE_SLOPE,
      " intercept", core.TRUE_INTERCEPT, " sigma", core.TRUE_SIGMA)
print("vibration classes  :", X_vib.shape, " counts", np.bincount(y_vib),
      " ", core.VIBRATION_CLASSES)
print("damped response    :", x_res.shape, " noiseless:",
      bool(np.allclose(y_res, core.damped_response(x_res))))

**What you should see.**

```
calibration line   : (40,)   true slope 2.4  intercept 0.8  sigma 0.35
vibration classes  : (360, 2)  counts [120 120 120]   ('balanced', 'imbalance', 'bearing fault')
damped response    : (200,)  noiseless: True
```

---

## 2 · The L-BFGS closure

Every optimiser you have used takes one gradient and one step. L-BFGS does not,
and every exercise in Part 2 uses it.

In [ ]:
# L-BFGS may evaluate the loss several times per step, so it needs a function
# it can call rather than a single backward pass. That function is the closure.
core.set_seed(0)
x, y = core.response_dataset(n=60)
X, Y = core.to_tensor(x), core.to_tensor(y)
loss_fn = nn.MSELoss()
model = core.MLP(hidden=(16, 16))
opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=20,
                        line_search_fn="strong_wolfe")

def closure():
    opt.zero_grad()                  # clear the last gradient
    loss = loss_fn(model(X), Y)      # forward, on all the data
    loss.backward()                  # backward
    return loss                      # L-BFGS reads the loss itself

for _ in range(10):
    opt.step(closure)                # may call closure() several times
with torch.no_grad():
    print(f"L-BFGS  loss after 10 steps: {loss_fn(model(X), Y).item():.5f}")

**What you should see.** A small finite loss; the exact number depends on
your PyTorch version.

What matters is the **shape of the call**. The closure does the whole
forward-and-backward, and returns the loss.
`opt.step` may call it several times inside a single step, because a quasi-Newton
method performs a **line search**: it decides on a direction, then tries
different distances along it and evaluates the loss at each. SGD, momentum and
Adam take a fixed step and never look back, so they need only one
evaluation.

Two consequences follow, and notebook 02 measures both.

**"One step" means different amounts of work for different optimisers.** Any
plot of loss against step number is comparing unequal things, and must say so.

**L-BFGS is full batch.** The line search compares losses, and a loss computed on
a different mini-batch each time is not comparable. This is why L-BFGS appears in
physics-informed work, where the whole problem fits in memory, and essentially
never in large-scale training.

---

## 3 · Ready — and which version to continue with

If every cell above ran, you are set up. Every numbered notebook in this set exists in **two versions with the same
text, the same figures and the same questions**. They differ only in whether
the code is already written.

| | continue with | the code |
|---|---|---|
| **write it yourself** | [`Ex06_01_loss_functions.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_01_loss_functions.ipynb) | cells marked `TODO` are yours to write |
| **read and run** | [`Ex06_01_loss_functions_light.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_01_loss_functions_light.ipynb) | complete: every cell runs as it stands |

Pick one and stay on it: each notebook links on to the next of the same kind,
and both tracks end at the same `Ex06_05_report.ipynb`. Switching is fine too - if a TODO
defeats you, open the same notebook's complete version, read that cell, and
carry on where you were. **The report is the same either way, and it is what
is marked**, so the complete version is not a shortcut past the work: it moves
the work from typing the code to reading it and explaining what it did.

A note on time. Every notebook in this set takes under a minute of compute on a
laptop CPU; notebook 02, the longest, about half a minute. Nothing needs a GPU.